# Into the Rabbit Hull — scaled-down reimplementation

Phase 1: DINOv2 activations → k-means centroids → trained stable SAE.

See `PAPER_NOTES.md` for the full paper extraction and `README.md` for what's scaled down and why. Downstream-task and geometry/MRH analysis are follow-up phases, listed at the bottom of this notebook.

## 0. Setup

In Colab: add an `HF_TOKEN` secret (Runtime → Secrets) with a HuggingFace token that has accepted the `imagenet-1k` dataset terms, then run the cells below.

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import sys
from pathlib import Path
sys.path.append("src")

import matplotlib.pyplot as plt

from config import CONFIG
import utils
import data_utils
import model_utils
import kmeans_utils
import sae as sae_module

print(CONFIG)

## 1. Load ImageNet-1k validation split and make the sae_train / analysis split

In [ ]:
dataset = data_utils.load_imagenet_val(CONFIG)
split = data_utils.make_split(len(dataset), CONFIG)
print({k: len(v) for k, v in split.items()})

## 2. Extract & cache DINOv2 activations

In [ ]:
model, processor = model_utils.load_model(CONFIG)

sae_train_images = [dataset[int(i)]["image"].convert("RGB") for i in split["sae_train"]]
sae_train_acts = utils.cached(
    Path(CONFIG["cache_dir"]) / "activations" / "sae_train.pt",
    lambda: model_utils.extract_activations(sae_train_images, model, processor, CONFIG),
)
sae_train_acts.shape

## 3. Fit k-means centroids (conv(A) approximation)

In [ ]:
centroids = utils.cached(
    Path(CONFIG["cache_dir"]) / "centroids.pt",
    lambda: kmeans_utils.fit_centroids(sae_train_acts, CONFIG),
)
centroids.shape

## 4. Train the stable SAE

In [ ]:
sae = sae_module.StableSAE(
    centroids, CONFIG["n_atoms"], CONFIG["sparsity_k"], CONFIG["d_model"]
)
history = sae_module.train_sae(sae, sae_train_acts, CONFIG)

In [ ]:
epochs = [h["epoch"] for h in history]
plt.plot(epochs, [h["r2"] for h in history])
plt.xlabel("epoch")
plt.ylabel("reconstruction R\u00b2")
plt.title("SAE reconstruction quality (paper reports >0.88 at full scale)")
plt.show()

## 5. Save the dictionary checkpoint

In [ ]:
sae_module.save_checkpoint(sae, CONFIG)

## What's next

This notebook covers `PAPER_NOTES.md` Sections 2 and 4 (SAE training). Follow-up phases, not yet implemented:

1. **Classification**: linear probe + concept importance (`PAPER_NOTES.md` §6) + Elsewhere concepts (§7).
2. **Segmentation (ADE20K)**: per-token linear probe + border concepts (§7).
3. **Dictionary geometry**: inner products, singular values, Hoyer scores, antipodal pairs, co-activation baselines (§9–11).
4. **Token-type-specific concepts**: footprint entropy, cls/reg/spatial-only concepts (§8).
5. **Position-embedding analysis** (§12).
6. **MRH empirical evidence**: k-NN geodesics, Archetypal Analysis, block structure (§14).